In [ ]:
import os
import numpy as np
import pandas as pd
import gc
from matplotlib import pyplot as plt

import dill

from dynamics_analysis_with_koopman.config.configs import PROJECT_ROOT
from dynamics_analysis_with_koopman.utils.checkpoint_registry import filter_eigenvalues_by_residual

KEEP_N = 25
CLUSTERING_METHOD = 'ward'  # complete, average, single, centroid, median, ward
NUM_TRIALS = 10000  # for the cluster significance
MODEL_FILE_PATH = 'edmd_model.pkl'

# Where this notebook's figures are stored, and in which format -- change
# FIGURE_FORMAT alone (e.g. to 'eps' or 'png') to switch every plot below.
ROOT_DIR = PROJECT_ROOT / 'papers' / 'clustering'
PLOTS_DIR = ROOT_DIR / 'wasserstein_clusters'
FIGURE_FORMAT = 'svg'
os.makedirs(PLOTS_DIR, exist_ok=True)

plt.rcdefaults()


def save_figure(name, fig=None):
    (fig or plt).savefig(os.path.join(PLOTS_DIR, f'{name}.{FIGURE_FORMAT}'), bbox_inches='tight')

## Load the eigenvalues and residuals from the trained models

In [ ]:
## load the fitted models
# The registry is written by papers/clustering/batch_run_main.py: one entry per
# experiment, holding the run ids of its preprocessing (step 1) and
# learning (step 3) run. The learning run is the one that fitted the model, so
# that is what is clustered here.
import json

RUN_REGISTRY = PROJECT_ROOT / 'papers' / 'clustering' / 'juelich_run_registry.json'

with open(RUN_REGISTRY) as f:
    registry = json.load(f)

runs_by_experiment = {
    experiment: entry for experiment, entry in sorted(registry['runs'].items()) if entry.get('learning_status') == 'ok'
}

skipped = [e for e in sorted(registry['runs']) if e not in runs_by_experiment]
assert len(skipped) == 0, "Not all runs in registry finished with status 'ok', stop the clustering here."

run_ids = [entry['learning_run_id'] for entry in runs_by_experiment.values()]
run_id_to_experiment = {entry['learning_run_id']: experiment for experiment, entry in runs_by_experiment.items()}
run_id_to_dir = {
    entry['learning_run_id']: PROJECT_ROOT / entry['learning_run_dir'] for entry in runs_by_experiment.values()
}

print(f'Number of runs to cluster: {len(run_ids)}')

In [ ]:
## define the labels for the experiment handles
# scenario type	scenario handle	geometry	flow directions	inflow	geometry adjusted	number of used trials	variation parameters	config file	clustering	num participants	grid shape	features/frame	comment	reference	perma link	download date
# bottleneck (one-off)	2006bottleneck 	bottleneck	unidirectional	one-off	-	-	geometry	2006bottleneck_dmd.yaml	-	[13, 13]	169	-	-	<link>	09.10.2024
RUN_OVERVIEW_CSV = ROOT_DIR / 'run_overview.csv'
header = [
    'scenario type',
    'scenario handle',
    'geometry',
    'flow directions',
    'inflow',
    'geometry adjusted',
    'number of used trials',
    'variation parameters',
    'config file',
    'clustering',
    'num participants',
    'grid shape',
    'features/frame',
    'comment',
    'reference',
    'perma link',
    'download date',
]
run_info = pd.read_csv(
    RUN_OVERVIEW_CSV,
    header=0,
)[header].dropna(subset=['scenario handle'])  # remove rows without a scenario handle # sep=r'\s*,'
run_info.columns = [col.replace(' ', '_') for col in run_info.columns]

In [ ]:
## create a latex table out of the columns needed
# one bibtex entry per unique perma_link (11 total)
# use link's dokuwiki page id as citation key
from datetime import datetime
from urllib.parse import urlparse, parse_qs


def _to_iso_date(download_date):
    return datetime.strptime(download_date, '%d.%m.%Y').strftime('%Y-%m-%d')


def _link_key(perma_link):
    return parse_qs(urlparse(perma_link).query)['id'][0]


link_groups = (
    run_info.assign(access_date=run_info['download_date'].apply(_to_iso_date))
    .groupby('perma_link')
    .agg(
        references=('reference', lambda refs: sorted(set(refs))),
        access_date=('access_date', 'max'),
    )
    .reset_index()
)
link_groups['bib_key'] = link_groups['perma_link'].apply(_link_key)


def _bibtex_entry(row):
    return (
        f'@misc{{{row["bib_key"]},\n'
        f'  author  = {{{{Forschungszentrum Jülich}}}},\n'
        f'  title   = {{Pedestrian Dynamics Data Archive: {", ".join(row["references"])}}},\n'
        f'  url     = {{{row["perma_link"]}}},\n'
        f'  urldate = {{{row["access_date"]}}},\n'
        f'  note    = {{Accessed {row["access_date"]}}}\n'
        f'}}'
    )


with open(f'{ROOT_DIR}/run_overview_references.bib', 'w') as f:
    f.write('\n\n'.join(link_groups.apply(_bibtex_entry, axis=1)) + '\n')

link_to_bib_key = dict(zip(link_groups['perma_link'], link_groups['bib_key']))
run_info['access_str_with_id'] = run_info.apply(
    lambda row: f'{row["scenario_handle"]} \\cite{{{link_to_bib_key[row["perma_link"]]}}}', axis=1
)

latex_columns = ['access_str_with_id', 'scenario_type', 'number_of_used_trials', 'comment']
header_aliases = ['Scenario handle', 'Scenario type', 'Trials', 'Summary']
with open(f'{ROOT_DIR}/run_overview_portrait.tex', 'w') as f:
    f.write(
        run_info[latex_columns].to_latex(
            header=header_aliases,
            index=False,
            caption='Overview of learned surrogate models',
            label='tab:run_overview',
            position='htbp',
            column_format='>{\\raggedright\\small\\hspace{0pt}}m{0.24\\linewidth}>{\\raggedright\\small\\hspace{0pt}}m{0.15\\linewidth}>{\\raggedright\\small\\hspace{0pt}}m{0.05\\linewidth}>{\\raggedright\\small\\arraybackslash}m{0.44\\linewidth}',
            multirow=True,
            escape=False,
            float_format='{:0.4f}'.format,
            longtable=True,
        )
    )

In [ ]:
## Load the eigenvalues for all run_ids
eigenvalue_data = {}
residual_data = {}
eigenvalue_index = {}
for run_id in run_ids:
    run_dir = run_id_to_dir[run_id]
    experiment = run_id_to_experiment[run_id]
    run_info.loc[(run_info['scenario_handle'] == experiment), 'run_id'] = run_id
    print(f'Found {experiment} files for {run_id=}')

    try:
        model_path = os.path.join(run_dir, 'models', MODEL_FILE_PATH)
        with open(model_path, 'rb') as f:
            edmd = dill.load(f)
        print('Model loaded successfully.')
    except FileNotFoundError as e:
        print(f'Model was not stored during creation of the checkpoint {run_id}.')
        print(f'{model_path=}')
        print('Aborting...')
        raise e

    eigenvalues = edmd.dmd_model.eigenvalues_
    eigenvalue_data[run_id] = eigenvalues
    eigenvalue_index[run_id] = list(range(eigenvalues.shape[0]))

    try:
        filter_target = edmd.dmd_model.residuals_
        assert len(eigenvalues) == len(filter_target), (
            'We can only use the filtering afterwards when we have all eigenvalues otherwise we cannot ensure index based matching.'
        )
        residual_data[run_id] = edmd.dmd_model.residuals_
    except AttributeError:
        try:
            filter_target = pd.read_csv(os.path.join(run_dir, 'data', 'residuals.csv')).values[0]
            residual_data[run_id] = filter_target
        except IndexError:
            print(f'Model residuals could not be found, no filtering is applied, loaded all eigenvalues for {run_id}')

    # remove the large model as a variable
    del edmd
    gc.collect()

## Plot the eigenvalues (filtered and unfiltered)

In [ ]:
## experiments to highlight in the eigenvalue/residual/permutation example plots below
TARGET_EXPERIMENTS = ['2009unidirClosed', '2013crossing90F', '2013unidirectionalB', '2022wheelchairU', '2017simeB']
experiment_to_run_id = {experiment: run_id for run_id, experiment in run_id_to_experiment.items()}
target_run_ids = [experiment_to_run_id[experiment] for experiment in TARGET_EXPERIMENTS]

# plot settings
TINY_SIZE = 8
SMALL_SIZE = 14
MEDIUM_SIZE = 16
BIGGER_SIZE = 20
plt.rc('font', size=SMALL_SIZE)  # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)  # fontsize of the axes title
plt.rc('axes', labelsize=BIGGER_SIZE)  # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)  # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title


def plot_eigenvalues_with_residuals(run_id, eigenvalue_data, residual_data, eigenvalue_index, marker_size=10):
    # the eigenvalues were already filtered when loading, so the stored indices
    # are used to pull the residuals belonging to exactly those eigenvalues
    keep_index = eigenvalue_index[run_id]
    eigenvalues = np.asarray(eigenvalue_data[run_id])
    residuals = np.asarray(residual_data[run_id])[keep_index]
    assert len(eigenvalues) == len(residuals), 'Eigenvalues and residuals do not match index wise.'

    fig = plt.figure(figsize=(4, 4))
    # add the unit circle
    theta = np.linspace(0, 2 * np.pi, 100)
    plt.plot(np.cos(theta), np.sin(theta), color='black', linewidth=1, label='Unit Circle')
    im = plt.scatter(
        x=eigenvalues.real,
        y=eigenvalues.imag,
        s=marker_size,
        zorder=5,
        cmap='jet',
        c=residuals,
        vmax=1,
        vmin=0,
    )

    ax = fig.axes[0]

    fig.subplots_adjust(right=0.85, top=0.85)
    fig.set_size_inches(4, 4 * 0.95)
    _, b, _, h = ax.get_position().bounds  # (left, bottom, width, height)
    cbar_ax = fig.add_axes((0.9, b, 0.01, h))
    fig.colorbar(im, cax=cbar_ax)

    ax.grid()
    # ax.set_xlim(.75, 1.05)
    # ax.set_ylim(-.28, .28)
    ax.set_title(f'{run_id}: {run_info.loc[run_info["run_id"] == run_id, "scenario_handle"].values[0]}\n')
    return fig, ax

In [ ]:
# plot the unfiltered eigenvalues
for scenario_id, run_id in zip(TARGET_EXPERIMENTS, target_run_ids):
    fig, ax = plot_eigenvalues_with_residuals(run_id, eigenvalue_data, residual_data, eigenvalue_index)

    save_figure(f'eigenvalues_{scenario_id}_no_filter', fig)

In [ ]:
## filter the eigenvalues using residuals
for run_id in run_ids:
    (eigenvalue_data[run_id], eigenvalue_index[run_id]) = filter_eigenvalues_by_residual(
        eigenvalue_data[run_id], residual_data[run_id], KEEP_N
    )

# plot the filtered eigenvalues
for scenario_id, run_id in zip(TARGET_EXPERIMENTS, target_run_ids):
    fig, ax = plot_eigenvalues_with_residuals(run_id, eigenvalue_data, residual_data, eigenvalue_index)

    ax.set_xlim(0.75, 1.05)
    ax.set_ylim(-0.28, 0.28)
    save_figure(f'eigenvalues_{scenario_id}_filtered', fig)

In [ ]:
## plot the distribution for the residuals for some example and the global one
from matplotlib import pyplot as plt

# plot the distribution of the residuals
for scenario_id, run_id in zip(TARGET_EXPERIMENTS, target_run_ids):
    res = residual_data[run_id]
    plt.hist(res, bins=50)
    plt.title(f'Distribution of residuals for {scenario_id}')
    plt.grid()
    save_figure(f'residuals_hist_{scenario_id}')
    plt.show()
    plt.close()

# plot the distribution of the residuals
global_residuals = np.array(list(residual_data.values())).flatten()
# counts, bins = np.histogram(global_residuals, bins=200)
plt.hist(global_residuals, bins=50)

plt.title(f'Distribution of residuals across {len(residual_data)} operators')
plt.grid()
save_figure('residuals_hist_global')
plt.show()

## Calculate the pair-wise distances using Wasserstein

In [ ]:
## get the distance matrix based on the selected measure
from scipy.linalg import issymmetric
import itertools

## For wasserstein based distance
# from https://github.com/william-redman/Identifying_Equivalent_Training_Dynamics/blob/main/Fully%20connected%20neural%20networks/wasserstein_distance.py
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment


def complex_wass_dist(e1, e2):
    # e1 and e2 are assumed to be two sets of complex eigenvalues
    s1 = np.vstack([np.real(e1), np.imag(e1)])
    s2 = np.vstack([np.real(e2), np.imag(e2)])
    d = cdist(s1.T, s2.T)
    assignment = linear_sum_assignment(
        d
    )  # matches eigenvalues such that the global! sum of pairwise distances is minimized
    return d[assignment].mean()  # , assignment


# calulcation of the wasserstein based distance matrix
# this minimizes the global sum over all pairs
org_distance_matrix = np.array(
    [complex_wass_dist(u, v) for u, v in list(itertools.product(eigenvalue_data.values(), eigenvalue_data.values()))]
).reshape(-1, len(run_ids))
print(f'Calulcated distance matrix using wasserstain distance for pairwise closest eigenvalues')

# sanity check
assert issymmetric(org_distance_matrix, rtol=1e-10, atol=1e-10), 'Check why the distance matrix is not symmetric!'

In [ ]:
## create clustering based on precomputed pairwise distance matrix
from matplotlib import pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

labels = [
    f'{run_info.loc[run_info["run_id"] == run_id, "scenario_handle"].values[0]}: '
    f'{run_info.loc[run_info["run_id"] == run_id, "scenario_type"].values[0]}'
    for run_id in run_ids
]  # use the labels in the same order as the org_linkage matrix

dists = squareform(org_distance_matrix, checks=False)

print(f'Obtained distance matrix of shape: {org_distance_matrix.shape}')
# print(org_distance_matrix[:5].mean(axis=0))

plt.imshow(np.sqrt(org_distance_matrix))  # sqrt(dists), the distances the linkage below uses
save_figure('distance_matrix')

In [ ]:
# check if the distance matrix is (squared) euclidean by double centering and checking the eigenvalues
# https://stats.stackexchange.com/a/13889 with https://stats.stackexchange.com/a/12503
# \cite{gower-1986-math}, Theorem 4

print(f'{org_distance_matrix.shape=}')
n_runs = org_distance_matrix.shape[0]
centering = np.eye(n_runs) - np.ones((n_runs, n_runs)) / n_runs


def is_squared_euclidean(dist_matrix, rtol=1e-10):
    sscp = -0.5 * centering @ dist_matrix @ centering
    evs = np.linalg.eigvalsh(sscp)  # symmetric, so the eigenvalues are real
    num_negative = (evs < -rtol * evs.max()).sum()
    return num_negative == 0, evs.min(), num_negative


for name, matrix in [
    ('squared euclidean (test on D)', org_distance_matrix),
    ('euclidean (test on D**2)', org_distance_matrix**2),
]:  # eleemtnwise
    passed, min_ev, num_negative = is_squared_euclidean(matrix)
    print(f'distance matrix is {name}? {passed} (min eigenvalue {min_ev:.2e}, {num_negative} negative)')

### Linkage Matrix
A (n-1) by 4 matrix Z is returned. At the i-th iteration, clusters with indices Z[i, 0] and Z[i, 1] are combined to form cluster n + i. A cluster with an index less than n corresponds to one of the n original observations. The distance between clusters Z[i, 0] and Z[i, 1] is given by Z[i, 2]. The fourth value Z[i, 3] represents the number of original observations in the newly formed cluster.

In [ ]:
# ward expects euclidean distances, the wasserstein distances are only squared euclidean,
# so the root is used: sqrt(dists) is euclidean and scipy squares it back internally
org_linkage_matrix = linkage(np.sqrt(dists), method=CLUSTERING_METHOD)
# columns: (node 1, node 2, distance between 1 and 2, number of observations in cluster)
print('First group of 3 is already created by the first 3 links:')
org_linkage_matrix[:3]

## Create the weighted graph layout
This allows us to visualize points relative to one another in space. 
We have no cooredinates for each node, just pair-wise distances.

In [ ]:
## visualize the data points in space as a weighted graph
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from scipy.spatial.distance import squareform
import pandas as pd

# linkage uses sqrt distances, so the layout shows the same distances
dist_matrix = np.sqrt(org_distance_matrix)
n_points = len(labels)
short_labels = [label.split(':')[0] for label in labels]

# Create a complete graph where edge weights are the distances
G = nx.Graph()
dist_dict = {}
for i in range(n_points):
    dist_dict[i] = {}
    for j in range(n_points):
        dist_dict[i][j] = dist_matrix[i, j]
        if i < j:
            G.add_edge(i, j, weight=dist_matrix[i, j])

# Compute layouts using Kamada-Kawai algorithm
pos_2d_graph = nx.kamada_kawai_layout(G, dist=dist_dict)

# --- Visualization ---
fig = plt.figure(figsize=(14, 7))

# Subplot 1: 2D Graph Layout without labels
ax1 = fig.add_subplot(121)
label_map = {i: '' for i, label in enumerate(short_labels)}
nx.draw(
    G,
    pos=pos_2d_graph,
    ax=ax1,
    with_labels=True,
    labels=label_map,
    node_color='lightgray',
    edgecolors='black',
    node_size=800,
    edge_color='white',
    alpha=0.7,
)
ax1.set_title('2D Graph Layout (Kamada-Kawai), no labels')

# Subplot 2: 2D Graph Layout with labels
ax3 = fig.add_subplot(122)
label_map = dict(enumerate(short_labels))  # all labels
nx.draw(
    G,
    pos=pos_2d_graph,
    ax=ax3,
    with_labels=True,
    labels=label_map,
    node_color='lightgray',
    edgecolors='black',
    node_size=800,
    edge_color='white',
    alpha=0.7,
)
ax3.set_title('2D Graph Layout (Kamada-Kawai)')
# rotatation is just for visualization, preserves pairwise distances
# plt.gca().invert_yaxis()
# plt.gca().invert_xaxis()

plt.tight_layout()
save_figure('2d-cluster-projection', fig)
plt.show()

## Test the significance for the clusters 

We shuffle the eigenvalues completely random across all experiments and cluster them with Ward linkage on the square root of the Wasserstein distance, same as the original data. 
Linkage significance is then based on the relative linkage height described in greenacre-2011-math.

In [ ]:
## get significance based on permutation testing and linkage height described in greenacre-2011-math
# shuffle the eigenvalues in the data
import random

random.seed(42)  # fixed seed, so the permutation p-values are reproducible

assert KEEP_N < 101
rand_linkage_heights = []
for _ in range(NUM_TRIALS):
    all_eigenvalues = np.array(list(eigenvalue_data.values())).flatten()  # returns a copy
    random.shuffle(all_eigenvalues)  # only truely shuffled when we have a 1d array (shuffles along first axis)
    all_eigenvalues = all_eigenvalues.reshape(len(run_ids), -1)

    perm_data = dict(zip(run_ids, all_eigenvalues))

    perm_dists = np.array(
        [complex_wass_dist(u, v) for u, v in list(itertools.product(perm_data.values(), perm_data.values()))]
    ).reshape(-1, len(run_ids))

    perm_dists = squareform(perm_dists, checks=False)
    links = linkage(
        np.sqrt(perm_dists), CLUSTERING_METHOD
    )  # same root as for org_linkage_matrix, so the heights are comparable
    rand_linkage_heights.append(links[:, 2])

# include the original in the p-value calculation
org_linkage_heights = org_linkage_matrix[:, 2]
rand_linkage_heights.append(org_linkage_heights)

# count how many permutation links are smaller than or equal to the original (per node)
p_values = ((rand_linkage_heights) <= org_linkage_heights).sum(axis=0) / len(rand_linkage_heights)

num_links = sum(p_values < 0.05)
print(
    f'Using clustering method {CLUSTERING_METHOD} on {KEEP_N} eigenvalues',
    f'filtered by residuals, the first {num_links} links are significantwhen using {NUM_TRIALS} permutations',
)
print(f'Longest significant link: {org_linkage_heights[num_links - 1]:.3f}')  # the links are ordered by length

np.set_printoptions(precision=6, suppress=True)
print(f'{p_values=}')

In [ ]:
## plot the example of one randomized set of eigenvalues
for scenario_id, run_id in zip(TARGET_EXPERIMENTS, target_run_ids):
    fig = plt.figure(figsize=(4, 4))
    # add the unit circle
    theta = np.linspace(0, 2 * np.pi, 100)
    plt.plot(np.cos(theta), np.sin(theta), color='black', linewidth=1, label='unit circle')
    im = plt.scatter(
        x=perm_data[run_id].real,
        y=perm_data[run_id].imag,
        s=10,
        zorder=5,
        cmap='jet',
        c=np.zeros_like(perm_data[run_id]),
        vmax=1,
        vmin=0,
    )

    ax = fig.axes[0]

    ax.grid()
    ax.set_xlim(0.75, 1.05)
    ax.set_ylim(-0.28, 0.28)
    ax.set_title(f'{run_id}: {scenario_id}')
    save_figure(f'permuted_eigenvalues_{scenario_id}', fig)

In [ ]:
from matplotlib.ticker import StrMethodFormatter

SIGNIFICANT_LINK_COLOR = 'darkslategray'

plt.close()

plt.figure(figsize=(14, 7))
plt.gca().xaxis.set_major_formatter(StrMethodFormatter('{x:,.2f}'))
plt.gca().tick_params(axis='x', labelsize=MEDIUM_SIZE)

significane_cutoff = (org_linkage_heights[num_links - 1] + org_linkage_heights[num_links]) / 2


def _link_color(node_id):
    # the significant links (below the cutoff) share one color, the rest stays gray
    if org_linkage_matrix[node_id - n_runs, 2] > significane_cutoff:
        return 'lightgray'
    return SIGNIFICANT_LINK_COLOR


with plt.rc_context({'lines.linewidth': 2}):  # scipy draws the links with the default line width (1.5)
    dendrogram(
        org_linkage_matrix,
        labels=labels,
        leaf_rotation=0,
        orientation='left',
        link_color_func=_link_color,
        leaf_font_size=16,
    )
plt.title(
    f'Clustering based on the {KEEP_N} eigenvalues, filtered by residuals, tested with {NUM_TRIALS} permutations\n',
)
plt.xlabel('Linkage distance', size=MEDIUM_SIZE)
plt.ylabel('Learned surrogate models', size=MEDIUM_SIZE)
plt.tight_layout()
save_figure('dendogram')
plt.show()

## Compare the clustering with groupings by experiment properties

We group the experiments based on the following categorical properties:
1. Geometry type
2. Inflow type (one-off vs. steady)
3. Uni- vs. bidirectional flow
4. Variation between the experiments: only the geometry changed, or the boundary conditions changed

In case an experiment cannot clearly be added to a category, it is excluded from that grouping.

For each grouping, and for the significant clusters of the unsupervised clustering (clusters with at least 2 runs),
we compute the silhouette coefficient on the same distances as the clustering (square root of the Wasserstein distance).
It compares the intra- and inter-cluster distances of each run and ranges from -1 to 1, best at 1.

In [ ]:
# header = ['scenario type', 'scenario handle', 'geometry', 'flow directions', 'inflow', 'geometry adjusted',
#           'number of used trials', 'variation parameters', 'config file', 'clustering', 'num participants',
#           'grid_shape', 'features/frame']

geometric_groups = run_info['geometry'].unique()
geometric_grouping = {
    g: list(run_info.loc[run_info['geometry'] == g, 'run_id'].values) for g in geometric_groups if g != '-'
}

flow_direction_groups = run_info['flow_directions'].unique()
uni_bi_grouping = {
    g: list(run_info.loc[run_info['flow_directions'] == g, 'run_id'].values) for g in flow_direction_groups if g != '-'
}

inflow_groups = run_info['inflow'].unique()
one_off_steady_grouping = {
    g: list(run_info.loc[run_info['inflow'] == g, 'run_id'].values) for g in inflow_groups if g != '-'
}

groups_geometry_adjusted = run_info['geometry_adjusted'].unique()
geometry_changes = {
    g: list(run_info.loc[run_info['geometry_adjusted'] == g, 'run_id'].values)
    for g in groups_geometry_adjusted
    if g != '-'
}

variation_groups = run_info['variation_parameters'].unique()
hyperparameter_changes = {
    g: list(run_info.loc[run_info['variation_parameters'] == g, 'run_id'].values) for g in variation_groups if g != '-'
}

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import fcluster


def get_unsupervised_clusters(linkage_matrix, run_ids, threshold, count_per_cluster):
    # 1. Flatten the linkage matrix into clusters based on the distance threshold
    cluster_labels = fcluster(linkage_matrix, t=threshold, criterion='distance')

    # 2. Identify clusters that meet the size requirement (size >= count_per_cluster)
    unique_labels, counts = np.unique(cluster_labels, return_counts=True)
    valid_clusters = unique_labels[counts >= count_per_cluster]

    # 3. Calculate Silhouette Score for the entire valid set
    # Note: Silhouette requires at least 2 clusters to be defined
    if len(valid_clusters) < 2:
        return 'Insufficient clusters (>=2) with size >= 2 to calculate silhouette score.'

    # Filter data to only include points in valid clusters for the score calculation
    mask = np.isin(cluster_labels, valid_clusters)
    if np.sum(mask) < 2:
        return 'Not enough data points in valid clusters.'

    # 4. Format output to match the 'hyperparameter_changes' structure
    # { cluster_id: [run_id1, run_id2, ... ] }
    cluster_groups = {
        int(cluster_id): [run_ids[i] for i in range(len(run_ids)) if cluster_labels[i] == cluster_id]
        for cluster_id in valid_clusters
    }

    return cluster_groups


unsupervised_clustering = get_unsupervised_clusters(
    org_linkage_matrix, run_ids, significane_cutoff, count_per_cluster=2
)
unsupervised_clustering

In [ ]:
from sklearn.metrics import silhouette_score, silhouette_samples

GROUPING = {
    'groups for experiment geometry': geometric_grouping,
    'groups for uni vs. bidirectional flow': uni_bi_grouping,
    'groups for one-off vs. steady inflow': one_off_steady_grouping,
    'group for changing only geometry': geometry_changes,
    'group for changing boundary conditions': hyperparameter_changes,
    'group from unsupervised clustering (>=2 runs per cluster)': unsupervised_clustering,
}

silhouette_scores = {}
for group_name, grouping in GROUPING.items():
    y_pred = []
    X = []
    for key, ids in grouping.items():
        for run_id in ids:
            evs = eigenvalue_data[run_id]
            evs_comp_stack = np.concatenate([np.real(evs), np.imag(evs)])
            X.append(evs_comp_stack)
            y_pred.append(key)

    # for exp, label in zip(X, y_pred):
    #     print(f"{exp}: {label}")
    def _real_complex_wass_dist(stacked_evs_1, stacked_evs_2):
        # the inputs are the stacked [real, imag] parts of two sets of complex eigenvalues
        s1 = stacked_evs_1.reshape(2, -1)
        s2 = stacked_evs_2.reshape(2, -1)
        d = cdist(s1.T, s2.T)
        assignment = linear_sum_assignment(d)  # matches the closest eigenvalues to one another
        # same root as for org_linkage_matrix, so the silhouette uses the distances the clustering used
        return np.sqrt(d[assignment].mean())

    # requires real values only, so we stack and unstack them accordingly in the metric function
    print('====')
    print(f'scores for {group_name} with {KEEP_N} eigenvalues, filtered by residuals:')
    pointwise_s_score = silhouette_samples(X, y_pred, metric=_real_complex_wass_dist)
    s_score = silhouette_score(X, y_pred, metric=_real_complex_wass_dist)
    silhouette_scores[group_name] = s_score
    print(f'  - silhouette score = {s_score:.3f}, (-1,1), best at 1')

In [ ]:
## write the silhouette table for the paper, rows in the order of the table
SILHOUETTE_TABLE_ROWS = {
    'group from unsupervised clustering (>=2 runs per cluster)': 'unsupervised clustering',
    'groups for experiment geometry': 'experiment geometry',
    'groups for uni vs. bidirectional flow': 'uni vs. bidirectional',
    'groups for one-off vs. steady inflow': 'one-off vs. steady inflow',
    'group for changing boundary conditions': 'changing boundary conditions',
    'group for changing only geometry': 'changing only geometry',
}
SILHOUETTE_TABLE_FILE = ROOT_DIR / 'silhouette_scores.tex'

rows = []
for group_name, row_label in SILHOUETTE_TABLE_ROWS.items():
    cluster_sizes = [len(ids) for ids in GROUPING[group_name].values()]
    rows.append(
        f'{row_label} & {len(cluster_sizes)} & \\numrange{{{min(cluster_sizes)}}}{{{max(cluster_sizes)}}} '
        f'& ${silhouette_scores[group_name]:.3f}$ \\\\'
    )

lines = [
    '\\begin{table}',
    '\\centering',
    '\\resizebox{\\columnwidth}{!}{%',
    '\\large',
    '\\begin{tabular}{>{\\raggedright\\arraybackslash}p{0.34\\columnwidth}|'
    '>{\\raggedright\\arraybackslash}p{0.2\\columnwidth}>{\\raggedright\\arraybackslash}p{0.4\\columnwidth}'
    '>{\\raggedright\\arraybackslash}p{0.27\\columnwidth}}',
    '\\toprule',
    '\\textbf{Clustering property} & \\textbf{No. of clusters} & \\textbf{Scenarios per cluster} & \\textbf{Silhouette coefficient} \\\\',
    '\\midrule',
    '\n\\midrule\n'.join(rows),
    '\\bottomrule',
    '\\end{tabular}',
    '}',
    '\\caption{Clustering silhouette coefficients for the unsupervised clustering in \\fref{fig:clustering-result} '
    'as well as clusters induced by single properties.}',
    '\\label{tab:silhouette-scores}',
    '\\end{table}',
]
silhouette_table = '\n'.join(lines) + '\n'
print(silhouette_table)